# Notebook 20 — Depth probe (G5)

Deeper 4-block CNN on the frozen split; frozen V-C score validity vs Fisher over all channel groups (G5a), and a 40%-realized calibrated 5-method comparison under minimal and standard recovery (G5b). Final experimental arm; resumable (teacher checkpoint, ablation CSV, regime CSV).

In [ ]:
# NB20 (G5 depth probe): deeper CNN on the frozen split; score validity + 40%-realized
# regime comparison. Pre-registered: G5a v_c>=fisher Spearman on >=2/4 harms (all groups);
# G5b minimal-recovery saber_v2 strictly best on awbir or family_f1, within 0.01 on the other.
from google.colab import drive; drive.mount("/content/drive", force_remount=False)
import os, sys, json
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

REPO = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
os.chdir(REPO); sys.path.insert(0, str(REPO))
from src.saber.bridge_ciciot import load_bridge
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.surgery import (enumerate_cnn1d_channel_groups, prune_cnn1d_channels,
                               profile_forward_flops, count_parameters)
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT = REPO / "results/saber/20_depth_probe"; OUT.mkdir(parents=True, exist_ok=True)
CKPT = REPO / "models/ciciot2023/deepcnn1d_g5_seed0.pt"
MIN_W = 8
torch.manual_seed(0); np.random.seed(0)

TRAIN_LOADER, VAL_LOADER, TEST_LOADER, _ANCHOR, CLASS_NAMES = load_bridge()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES)
robust_graph = pd.read_csv(REPO / "results/saber/14_risk_graph/asvg_edges_robust.csv")
N_CLASSES = len(CLASS_NAMES)

class DeepCNN1D(nn.Module):
    def __init__(self, n_classes=34):
        super().__init__()
        def blk(i, o): return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(o)]
        self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2),
                                  *blk(128, 128), *blk(128, 256))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(256, n_classes)
    def forward(self, x):
        if x.dim() == 2: x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))

train_y = TRAIN_LOADER.dataset.tensors[1].numpy()
counts = np.bincount(train_y, minlength=N_CLASSES)
w = np.zeros_like(counts, float); w[counts > 0] = 1/np.sqrt(counts[counts > 0])
w[counts > 0] /= w[counts > 0].mean()
CLASS_W = torch.tensor(w, dtype=torch.float32, device=DEVICE)

def train(model, loader, epochs, lr=1e-3):
    model = model.to(DEVICE).train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lf = nn.CrossEntropyLoss(weight=CLASS_W)
    for ep in range(epochs):
        for x, y in loader:
            opt.zero_grad(); lf(model(x.to(DEVICE)), y.to(DEVICE)).backward(); opt.step()
        print(f"  epoch {ep+1}/{epochs} done")
    return model.eval()

if CKPT.exists():
    TEACHER = DeepCNN1D(N_CLASSES); TEACHER.load_state_dict(
        torch.load(CKPT, map_location="cpu", weights_only=False)["state_dict"])
    TEACHER = TEACHER.to(DEVICE).eval(); print("teacher loaded from cache")
else:
    print("training deep teacher (4 epochs)...")
    TEACHER = train(DeepCNN1D(N_CLASSES), TRAIN_LOADER, epochs=4)
    torch.save({"state_dict": TEACHER.cpu().state_dict()}, CKPT); TEACHER = TEACHER.to(DEVICE)

def logits_of(model, loader):
    model = model.to(DEVICE).eval(); outs = []
    with torch.no_grad():
        for x, _ in loader: outs.append(model(x.to(DEVICE)).cpu().numpy())
    return np.concatenate(outs)

VAL_Y = VAL_LOADER.dataset.tensors[1].numpy()
T_VAL = logits_of(TEACHER, VAL_LOADER)
t_audit = full_model_audit(T_VAL, VAL_Y, taxonomy, DEFAULT_COST_PROFILES)
print("dense deep teacher:", {k: round(float(t_audit[k]), 4) for k in
      ["fine_macro_f1", "family_macro_f1", "attack_to_benign_rate", "benign_to_attack_rate"]})

EX = next(iter(VAL_LOADER))[0][:8].float().to(DEVICE)
groups = enumerate_cnn1d_channel_groups(TEACHER, EX)
print("prunable groups:", len(groups), "| layers:",
      groups.groupby("module_path")["group_id"].count().to_dict())

# ---- balanced ablation subset (512/class, seed 0) ----
Xv, Yv = VAL_LOADER.dataset.tensors
idx = []
rng = np.random.default_rng(0)
for c in range(N_CLASSES):
    ci = np.where(VAL_Y == c)[0]
    idx.append(rng.permutation(ci)[:512])
idx = np.concatenate(idx)
SX, SY = Xv[idx].to(DEVICE), VAL_Y[idx]
with torch.no_grad(): T_SUB = TEACHER(SX).cpu().numpy()
ts_audit = full_model_audit(T_SUB, SY, taxonomy, DEFAULT_COST_PROFILES)

# ---- scores (self-contained) ----
conv_paths = sorted(groups["module_path"].unique())
def conv_w(path):
    return dict(TEACHER.named_modules())[path].weight

mag = {p: conv_w(p).detach().pow(2).sum(dim=(1, 2)).sqrt().cpu().numpy() for p in conv_paths}
tay = {p: np.zeros(conv_w(p).shape[0]) for p in conv_paths}
fis = {p: np.zeros(conv_w(p).shape[0]) for p in conv_paths}
lf = nn.CrossEntropyLoss(weight=CLASS_W)
nb = 0
for x, y in VAL_LOADER:
    TEACHER.zero_grad(); lf(TEACHER(x.to(DEVICE)), y.to(DEVICE)).backward()
    for p in conv_paths:
        wt, g = conv_w(p), conv_w(p).grad
        tay[p] += (wt * g).sum(dim=(1, 2)).abs().detach().cpu().numpy()
        fis[p] += (wt * g).pow(2).sum(dim=(1, 2)).detach().cpu().numpy()
    nb += 1
    if nb >= 40: break
TEACHER.zero_grad()

# semantic boundary leverage per group (robust-weight graph, frozen simplification)
sbl = {p: np.zeros(conv_w(p).shape[0]) for p in conv_paths}
for _, e in robust_graph.iterrows():
    s, t, ew = int(e["source_index"]), int(e["target_index"]), float(e["robust_weight"])
    rows = np.where(SY == s)[0][:256]
    if len(rows) == 0: continue
    TEACHER.zero_grad()
    z = TEACHER(SX[rows])
    (z[:, s] - z[:, t]).mean().backward()
    for p in conv_paths:
        wt, g = conv_w(p), conv_w(p).grad
        sbl[p] += ew * (wt * g).sum(dim=(1, 2)).abs().detach().cpu().numpy()
TEACHER.zero_grad()

sc = groups[["group_id", "module_path", "channel_index", "flops_cost"]].copy()
for name, d in [("magnitude", mag), ("taylor", tay), ("fisher", fis), ("sbl_raw", sbl)]:
    sc[name] = [d[r.module_path][int(r.channel_index)] for r in sc.itertuples()]
sc["sem_rank"] = sc.groupby("module_path")["sbl_raw"].rank(pct=True)
layer_f = sc.groupby("module_path")["fisher"].mean()
sc["v_c"] = sc["sem_rank"] * sc["module_path"].map(layer_f)
rng2 = np.random.default_rng(1); sc["random"] = rng2.random(len(sc))
sc.to_csv(OUT / "deep_group_scores.csv", index=False)

# ---- G5a: per-group functional ablation harm (resumable) ----
HARM_CSV = OUT / "deep_group_harm.csv"
done_rows = pd.read_csv(HARM_CSV).to_dict("records") if HARM_CSV.exists() else []
done_ids = {r["group_id"] for r in done_rows}
mods = dict(TEACHER.named_modules())
bn_of = dict(zip(groups["module_path"], groups.get("batchnorm_path", [None]*len(groups)))) \
        if "batchnorm_path" in groups.columns else {}
for k, r in enumerate(groups.itertuples()):
    if r.group_id in done_ids: continue
    conv = mods[r.module_path]; c = int(r.channel_index)
    saved = [conv.weight.data[c].clone(), conv.bias.data[c].clone() if conv.bias is not None else None]
    conv.weight.data[c] = 0
    if conv.bias is not None: conv.bias.data[c] = 0
    bnp = bn_of.get(r.module_path)
    if bnp:
        bn = mods[bnp]
        sb = [bn.weight.data[c].clone(), bn.bias.data[c].clone()]
        bn.weight.data[c] = 0; bn.bias.data[c] = 0
    with torch.no_grad(): sl = TEACHER(SX).cpu().numpy()
    a = full_model_audit(sl, SY, taxonomy, DEFAULT_COST_PROFILES)
    aw, _ = action_weighted_boundary_inversion_rate(T_SUB, sl, SY, robust_graph)
    done_rows.append({"group_id": r.group_id, "module_path": r.module_path,
        "channel_index": c, "harm_awbir": float(aw),
        "harm_fine_macro_f1": float(ts_audit["fine_macro_f1"] - a["fine_macro_f1"]),
        "harm_family_macro_f1": float(ts_audit["family_macro_f1"] - a["family_macro_f1"]),
        "harm_hsr_balanced_soc": float(a["hsr_balanced_soc"] - ts_audit["hsr_balanced_soc"])})
    conv.weight.data[c] = saved[0]
    if saved[1] is not None: conv.bias.data[c] = saved[1]
    if bnp: bn.weight.data[c] = sb[0]; bn.bias.data[c] = sb[1]
    if (k + 1) % 25 == 0:
        pd.DataFrame(done_rows).to_csv(HARM_CSV, index=False)
        print(f"ablation {k+1}/{len(groups)}")
pd.DataFrame(done_rows).to_csv(HARM_CSV, index=False)
harm = pd.DataFrame(done_rows).merge(sc, on=["group_id", "module_path", "channel_index"])
HK = ["harm_awbir", "harm_fine_macro_f1", "harm_family_macro_f1", "harm_hsr_balanced_soc"]
g5a_tab = {s: {h: float(spearmanr(harm[s], harm[h]).correlation) for h in HK}
           for s in ["v_c", "fisher", "taylor", "magnitude"]}
g5a_wins = sum(g5a_tab["v_c"][h] >= g5a_tab["fisher"][h] for h in HK)
print("\nG5a Spearman table:", json.dumps(g5a_tab, indent=1))
print("G5a: v_c >= fisher on", g5a_wins, "/4 -> pass:", g5a_wins >= 2)

# ---- G5b: 40% realized, minimal + standard recovery ----
METHODS = {"random": "random", "magnitude": "magnitude", "taylor": "taylor",
           "fisher": "fisher", "saber_v2": "v_c"}
def order_of(col):
    t = sc.sort_values(col, ascending=True)
    left = {p: int((sc["module_path"] == p).sum()) for p in conv_paths}
    seq = []
    for r in t.itertuples():
        if left[r.module_path] - 1 < MIN_W: continue
        left[r.module_path] -= 1; seq.append((r.module_path, int(r.channel_index)))
    return seq
def prune_prefix(seq, k):
    pm = {}
    for p, c in seq[:k]: pm.setdefault(p, []).append(c)
    pm = {p: sorted(cs) for p, cs in pm.items()}
    st, _ = prune_cnn1d_channels(TEACHER, pm, EX, minimum_remaining_per_layer=MIN_W)
    return st.to(DEVICE)
M0F = profile_forward_flops(TEACHER, EX)["flops_per_item"]
def rr(st): return 1 - profile_forward_flops(st, EX)["flops_per_item"] / M0F
def calib(seq, target=0.40):
    lo, hi = 1, len(seq)
    if rr(prune_prefix(seq, hi)) < target: return hi
    while lo < hi:
        mid = (lo + hi) // 2
        if rr(prune_prefix(seq, mid)) >= target: hi = mid
        else: lo = mid + 1
    return lo

g = torch.Generator().manual_seed(0)
sub_i = torch.randperm(len(TRAIN_LOADER.dataset), generator=g)[: len(TRAIN_LOADER.dataset)//10]
SUB = torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, sub_i.tolist()),
      batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(0))
def audit_val(st):
    L = logits_of(st, VAL_LOADER)
    a = full_model_audit(L, VAL_Y, taxonomy, DEFAULT_COST_PROFILES)
    aw, _ = action_weighted_boundary_inversion_rate(T_VAL, L, VAL_Y, robust_graph)
    a["awbir"] = float(aw); return a

RES = OUT / "depth_regime_results.csv"
rows = pd.read_csv(RES).to_dict("records") if RES.exists() else []
done2 = {(r["method"], r["regime"]) for r in rows}
for m, col in METHODS.items():
    seq = order_of(col); k = calib(seq)
    for regime, fn in [("minimal", lambda s: train(s, SUB, 1)),
                       ("standard", lambda s: train(s, TRAIN_LOADER, 2))]:
        if (m, regime) in done2: continue
        torch.manual_seed(0)
        st = prune_prefix(seq, k)
        realized = rr(st)
        st = fn(st)
        a = audit_val(st)
        rows.append({"method": m, "regime": regime, "k": k, "realized": float(realized),
            "parameters": count_parameters(st),
            **{key: float(a[key]) for key in ["awbir", "fine_macro_f1", "family_macro_f1",
               "attack_to_benign_rate", "benign_to_attack_rate", "hsr_balanced_soc", "ece15"]}})
        pd.DataFrame(rows).to_csv(RES, index=False)
        print(m, regime, "k=", k, "realized=", round(float(realized), 3),
              "awbir=", round(rows[-1]["awbir"], 4), "famF1=", round(rows[-1]["family_macro_f1"], 4))

df = pd.DataFrame(rows)
mn = df[df["regime"] == "minimal"].set_index("method")
best_aw = mn["awbir"].idxmin(); best_ff = mn["family_macro_f1"].idxmax()
cond_aw = best_aw == "saber_v2" and (mn["family_macro_f1"].max() - mn.loc["saber_v2", "family_macro_f1"]) <= 0.01
cond_ff = best_ff == "saber_v2" and (mn.loc["saber_v2", "awbir"] - mn["awbir"].min()) <= 0.01
g5b = bool(cond_aw or cond_ff)
std = df[df["regime"] == "standard"].set_index("method")
gate = {"gate": "G5_depth_probe", "passed": bool(g5a_wins >= 2 and g5b),
        "g5a_wins_vs_fisher": int(g5a_wins), "g5a_table": g5a_tab,
        "g5b_passed": g5b, "g5b_best_awbir": best_aw, "g5b_best_family_f1": best_ff,
        "standard_awbir_spread": float(std["awbir"].max() - std["awbir"].min()),
        "criterion": "G5a: v_c>=fisher on >=2/4 harms (all groups). G5b: minimal-recovery "
                     "saber_v2 strictly best on awbir or family_f1 and within 0.01 on the other."}
(OUT / "G5_depth_gate.json").write_text(json.dumps(gate, indent=2))
print(json.dumps(gate, indent=2))
print(df.sort_values(["regime", "method"]).to_string(index=False))


In [ ]:
print("hi")

In [ ]:
from pathlib import Path
import json, pandas as pd
O = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/results/saber/20_depth_probe")
print((O / "G5_depth_gate.json").read_text())
print(pd.read_csv(O / "depth_regime_results.csv").sort_values(["regime", "method"]).to_string(index=False))